# Colab Scrape — Remaining 2,485 GUIDs (different IP)

Mac IP rate-limited by TBMM after ~44h scraping. Continuing remaining 2.5K GUIDs from Colab (fresh IP).

**Prerequisites**
- Upload `data/bronze/guids_remaining.txt` to Drive at `/MyDrive/stat401-tbmm/bronze/`.
- Empty target dir: `/MyDrive/stat401-tbmm/bronze/colab_meta_shards/`.

**Outputs**
- `bronze/colab_meta_shards/part-NNNN.parquet` — Parquet shards (500 rows each).
- `bronze/colab_done_meta.txt` — resume key (newline GUIDs).
- `bronze/colab_scraper_errors.parquet` — failures.

**ETA** ~42 min at 1 sec/önerge.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
!pip install -q tenacity

In [ ]:
from __future__ import annotations
import hashlib, time, traceback
from dataclasses import asdict, dataclass
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tenacity import retry, stop_after_attempt, wait_exponential
from tqdm.auto import tqdm

BASE = 'https://www.tbmm.gov.tr'
DETAIL_TPL = f'{BASE}/Denetim/Yazili-Soru-Onergesi-Detay/{{guid}}'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0 Safari/537.36',
    'Accept-Language': 'tr-TR,tr;q=0.9,en;q=0.8',
}

PROJECT = Path('/content/drive/MyDrive/stat401-tbmm')
GUIDS_FILE = PROJECT / 'bronze' / 'guids_remaining.txt'
SHARD_DIR = PROJECT / 'bronze' / 'colab_meta_shards'
DONE_FILE = PROJECT / 'bronze' / 'colab_done_meta.txt'
ERR_FILE = PROJECT / 'bronze' / 'colab_scraper_errors.parquet'

SHARD_DIR.mkdir(parents=True, exist_ok=True)
DONE_FILE.parent.mkdir(parents=True, exist_ok=True)

RATE_SEC = 1.0
BATCH_SIZE = 500

In [ ]:
@dataclass
class Onerge:
    guid: str
    donem: str
    yasama_yili: str
    esas_no: str
    geliş_tarihi: str
    özet: str
    sahip_il: str
    sahip_isim: str
    muhatap_bakanlık: str
    muhatap_bakan: str
    durum: str
    onerge_pdf_url: str | None
    cevap_pdf_url: str | None
    detail_html_hash: str

session = requests.Session()
session.headers.update(HEADERS)
_last = [0.0]

@retry(stop=stop_after_attempt(5), wait=wait_exponential(min=2, max=60))
def fetch_html(url: str) -> str:
    now = time.monotonic()
    gap = now - _last[0]
    if gap < RATE_SEC:
        time.sleep(RATE_SEC - gap)
    _last[0] = time.monotonic()
    r = session.get(url, timeout=30)
    if r.status_code == 429:
        ra = int(r.headers.get('Retry-After', 30))
        time.sleep(ra)
        raise requests.HTTPError(f'429 — sleeping {ra}s')
    r.raise_for_status()
    return r.text

def parse_detail(guid: str) -> Onerge:
    html = fetch_html(DETAIL_TPL.format(guid=guid))
    h = hashlib.sha256(html.encode()).hexdigest()[:16]
    soup = BeautifulSoup(html, 'lxml')
    fields = {}
    for row in soup.select('tr'):
        cells = [c.get_text(strip=True) for c in row.find_all(['td', 'th'])]
        if len(cells) == 2:
            fields[cells[0]] = cells[1]
    f = lambda k: fields.get(k, '').strip()

    sahip_raw = f('Önergenin Sahibi'); sahip_il, sahip_isim = '', sahip_raw
    if ' Milletvekili ' in sahip_raw:
        sahip_il, sahip_isim = sahip_raw.split(' Milletvekili ', 1)

    muh_raw = f('Önergenin Muhatabı'); muh_bak, muh_isim = muh_raw, ''
    if ' Bakanı ' in muh_raw:
        muh_bak, muh_isim = muh_raw.split(' Bakanı ', 1)
        muh_bak = muh_bak + ' Bakanlığı'

    pdf_urls = [urljoin(BASE, a['href']) for a in soup.select("a[href$='.pdf'], a[href*='.pdf?']")]
    onerge_pdf = pdf_urls[0] if pdf_urls else None
    cevap_pdf = pdf_urls[1] if len(pdf_urls) > 1 else None

    donem_str = f('Dönemi ve Yasama Yılı')
    donem, yyil = (donem_str.split('/') + ['', ''])[:2]

    return Onerge(
        guid=guid, donem=donem.strip(), yasama_yili=yyil.strip(),
        esas_no=f('Esas Numarası'),
        geliş_tarihi=f('Başkanlığa Geliş Tarihi'),
        özet=f('Önergenin Özeti'),
        sahip_il=sahip_il.strip(), sahip_isim=sahip_isim.strip(),
        muhatap_bakanlık=muh_bak.strip(), muhatap_bakan=muh_isim.strip(),
        durum=f('Son Durumu'),
        onerge_pdf_url=onerge_pdf, cevap_pdf_url=cevap_pdf,
        detail_html_hash=h,
    )

In [ ]:
guids = [g.strip() for g in GUIDS_FILE.read_text().splitlines() if g.strip()]
print(f'TODO: {len(guids)} GUIDs')

done = set(DONE_FILE.read_text().splitlines()) if DONE_FILE.exists() else set()
todo = [g for g in guids if g not in done]
print(f'after resume: {len(todo)} left')

In [ ]:
existing = sorted(SHARD_DIR.glob('part-*.parquet'))
next_idx = int(existing[-1].stem.split('-')[1]) + 1 if existing else 0
print(f'next shard: {next_idx:04d}')

def flush(rows, idx):
    if not rows: return
    p = SHARD_DIR / f'part-{idx:04d}.parquet'
    pd.DataFrame(rows).to_parquet(p, index=False)
    print(f'flushed {len(rows)} → {p.name}')

def append_done(keys):
    with DONE_FILE.open('a') as f:
        for k in keys: f.write(k + '\n')

def append_err(errs):
    if not errs: return
    new = pd.DataFrame(errs)
    if ERR_FILE.exists():
        new = pd.concat([pd.read_parquet(ERR_FILE), new], ignore_index=True)
    new.to_parquet(ERR_FILE, index=False)

buffer, flushed, errors = [], [], []
with tqdm(total=len(todo), desc='scrape', unit='onerge') as pbar:
    for g in todo:
        try:
            o = parse_detail(g)
            buffer.append(asdict(o))
            flushed.append(g)
        except Exception as e:
            errors.append({'guid': g, 'error': repr(e)[:300]})
        pbar.update(1)
        if len(flushed) >= BATCH_SIZE:
            flush(buffer, next_idx)
            append_done(flushed)
            append_err(errors)
            next_idx += 1
            buffer, flushed, errors = [], [], []

flush(buffer, next_idx)
append_done(flushed)
append_err(errors)
print('done')

In [ ]:
shards = sorted(SHARD_DIR.glob('part-*.parquet'))
df = pd.concat([pd.read_parquet(s) for s in shards], ignore_index=True)
print(f'colab shards: {len(shards)}, total rows: {len(df)}')
print(df.head())
df.to_parquet(PROJECT / 'bronze' / 'colab_meta.parquet', index=False)
print(f'wrote bronze/colab_meta.parquet')